# 06 Разбор отладки ROCm и кейсы для устранения неполадок

        Этот раздел превращает путь проб и ошибок при воспроизведении в рабочую книгу по устранению неполадок. Не нужно помнить все команды: важно разложить симптом на доказательства, корневую причину, исправление и проверку.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("Запустите Jupyter из каталога темы AMD ROCm или из подкаталога notebooks.")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"Нет файла материала: {path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1: шаблон записи о неполадке


In [ ]:
template = """
### Название проблемы

- Симптом:
- Команда воспроизведения:
- Ключевые доказательства:
- Что исключено:
- Корневая причина:
- Исправление:
- Проверка после исправления:
- Вывод для обучения:
"""
print(template)


## Checkpoint 2: индекс девяти типичных кейсов


In [ ]:
rows = [
    ("Ошибочная оценка старого success", "нужно ввести physical_success"),
    ("Проблема меток gripper в демонстрациях", "сначала проверить данные, потом подозревать модель"),
    ("ACT: loss падает, а closed-loop проваливается", "смотреть open-loop и closed-loop отдельно"),
    ("После добавления данных DAgger качество падает", "в новых данных указать, какое распределение состояний они дополняют"),
    ("Смещение SmolVLA в сторону красной или синей чашки", "оценивать отдельно по instruction"),
    ("Копирование данных вредит задаче другого типа", "сначала пробовать взвешивание на уровне sampler или loss"),
    ("Видео и статистика batch не совпадают", "видео объясняет поведение, успешность смотрим по batch JSONL"),
    ("Проблемы ресурсов при длительном обучении ROCm", "скриптовать, оценивать пакетами, проверять процессы и температуру"),
    ("Ограничение перед обучением pi0", "о длительном обучении — только после доступа и smoke"),
    ("Остатки reset у pi0", "перед непрерывной оценкой в одном окружении сначала очистить состояние динамики MuJoCo"),
]
md_table(["Кейс", "Вывод для обучения"], rows)


## Checkpoint 3: кейс протокола оценки pi0 с hard-reset


Один случай отладки финишера `dynamic_timed` очень типичен: после исправления state с учётом стадии старая непрерывная оценка уже проходила strict `30/30`, но у seed `1036` конечное `xy` однажды подошло близко к порогу. При дальнейшей трассировке выяснилось, что начальное положение чашки для следующего seed иногда выходило за пределы диапазона сэмплирования этого seed, то есть состояние qvel / ctrl / free-joint предыдущего episode загрязнило следующий episode.

        Здесь не стоит сразу считать это проблемой генерализации модели. Более чистый путь — сначала исправить протокол оценки: для малой панели можно использовать `--fresh-env-per-episode`, создавая новое окружение для каждого seed; для полной пакетной оценки рекомендуется `--hard-reset-sim-data`, который очищает нижележащие данные симуляции MuJoCo перед каждым `env.reset(seed)`. После повторного прогона seed сэмплирования политики `1010-1039` (окружение фактически фиксировано на seed 0) в чистом протоколе hard-reset результат остаётся strict `30/30`, legacy `30/30`, mean `xy=0.0219 m`, max `xy=0.0450 m`, а seed `1036` больше не находится на границе.


In [ ]:
rows = [
    ("Симптом", "старая непрерывная оценка 30/30, но у seed 1036 однажды max xy=0.0993 m"),
    ("Аномалия", "у seed 1035 начальное положение чашки оказалось вне диапазона сэмплирования"),
    ("Корневая причина", "env.reset(seed) переустанавливает положения объектов, но не очищает остаточную динамику MuJoCo"),
    ("Исправление", "добавлены --fresh-env-per-episode и --hard-reset-sim-data"),
    ("Чистый результат", "после hard-reset 1010-1039: strict 30/30, mean xy=0.0219 m, max xy=0.0450 m"),
]
md_table(["Пункт", "Запись"], rows)


## Checkpoint 4: смотрим метрики и видео вместе


In [ ]:
show_asset("act_dagger_progress_curve.png", width=900)
show_asset("smolvla_red_blue_success.png", width=1000)


График отвечает на вопрос «каков общий результат», а видео — на вопрос «действительно ли поведение похоже на успех». Оба типа доказательств лучше одновременно включать в отчёт об эксперименте.


## Checkpoint 5: шаблон сводки результатов


In [ ]:
rows = [
    ("Устройство", "модель GPU/APU, ROCm, PyTorch, температура и видеопамять"),
    ("Данные", "число episode, тип задачи, прошёл ли аудит физического воспроизведения"),
    ("ACT", "best checkpoint, physical_success, основные типы провалов"),
    ("SmolVLA", "успешность для красной и синей чашки, стратегия сэмплирования, защищённый baseline"),
    ("pi0", "доступ gated, 1-step smoke, статус полного обучения"),
    ("Видео", "один реальный успех и один типичный провал"),
    ("Разбор", "самая важная ловушка и доказательство исправления"),
]
md_table(["Пункт", "Что нужно написать"], rows)
